# 01 — Análisis exploratorio
Detección de tala selectiva en Madre de Dios (Sentinel-2 + Deep Learning)

Este notebook está pensado para correr en **Google Colab**. Objetivos:
1. Autenticar Earth Engine.
2. Visualizar el ROI (corredor Iberia–Iñapari) y un compuesto Sentinel-2.
3. Calcular y visualizar dNDVI/dNBR entre dos periodos.
4. Explorar histogramas de dNDVI/dNBR para ajustar los umbrales de la pseudo-etiqueta (`src/labels.py`).
5. Generar la grilla de export y lanzar un export de PRUEBA (pocos tiles) a Drive.

## 0. Setup (solo Colab)

In [ ]:
# Si corres esto en Colab, descomenta para clonar/instalar:
 !git clone <https://github.com/CristopherVQ1024/GEE_API>.git
 %cd madre_de_dios_logging_detection
 !pip install -q -r requirements.txt

import sys
sys.path.append('..')  # para poder hacer `import src....` si el notebook corre desde notebooks/

In [ ]:
import ee
import geemap

from src.gee_exporter import (
    load_config, init_ee, get_roi_geometry, get_sentinel2_composite,
    build_feature_stack, compute_change_layers, build_export_grid,
)
from src import labels

cfg = load_config('../config/config.yaml')
print('Recuerda editar config/config.yaml -> gee.project_id con tu Cloud Project registrado.')
print('Registro: https://code.earthengine.google.com/register')
init_ee(cfg)
print('Earth Engine inicializado correctamente ✅')

## 1. Área de estudio

In [ ]:
roi = get_roi_geometry(cfg)

Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(roi, {'color': 'blue'}, 'ROI: corredor Iberia-Iñapari')
Map

**Antes de seguir**: mira el mapa. ¿El ROI cubre bien el corredor y el patrón de
fragmentación/tala que te interesa? Si no, ajusta `roi.bbox` en `config.yaml` y
vuelve a correr esta celda.

## 2. Compuesto Sentinel-2 (última ventana temporal)

In [ ]:
last_window = cfg['temporal_windows'][-1]
composite = get_sentinel2_composite(cfg, roi, last_window['start_date'], last_window['end_date'])

vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}
Map.addLayer(composite, vis_rgb, f"S2 RGB {last_window['id']}")
Map

## 3. Stack de 8 bandas (features del modelo)

In [ ]:
stack = build_feature_stack(cfg, roi, last_window['start_date'], last_window['end_date'])
print('Bandas del stack:', stack.bandNames().getInfo())

vis_ndvi = {'bands': ['NDVI'], 'min': -0.2, 'max': 0.9, 'palette': ['brown', 'yellow', 'green']}
Map.addLayer(stack, vis_ndvi, 'NDVI')
Map

## 4. Diferencias multitemporales (dNDVI, dNBR)

In [ ]:
pair = cfg['default_change_pair']
change = compute_change_layers(cfg, roi, pair['before'], pair['after'])

vis_change = {'bands': ['dNDVI'], 'min': -0.5, 'max': 0.1, 'palette': ['red', 'white', 'green']}
Map.addLayer(change, vis_change, f"dNDVI ({pair['before']} -> {pair['after']})")
Map

## 5. Histogramas de dNDVI/dNBR (para calibrar umbrales de `labels.py`)
Extraemos una muestra de píxeles dentro del ROI para inspeccionar la distribución
y elegir umbrales razonables (ver `labels.suggest_change_thresholds_from_sample`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample_fc = change.sample(region=roi, scale=10, numPixels=5000, seed=42, geometries=False)
sample_list = sample_fc.reduceColumns(ee.Reducer.toList(2), ['dNDVI', 'dNBR']).get('list').getInfo()
sample_arr = np.array(sample_list)
dndvi_vals, dnbr_vals = sample_arr[:, 0], sample_arr[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(dndvi_vals, bins=60); axes[0].set_title('Distribución dNDVI')
axes[1].hist(dnbr_vals, bins=60); axes[1].set_title('Distribución dNBR')
plt.show()

suggested = labels.suggest_change_thresholds_from_sample(dndvi_vals, dnbr_vals, percentile=5.0)
print('Umbrales sugeridos (percentil 5):', suggested)
print('Compáralos con los valores por defecto en labels.compute_disturbance_proxy_mask')

## 6. Pseudo-etiqueta de disturbio (heurística)

In [ ]:
proxy_mask = labels.build_training_label_image(
    hansen_asset_id=cfg['reference_datasets']['hansen_gfc']['asset_id'],
    change_image=change,
    ndvi_drop_threshold=suggested['ndvi_drop_threshold'],
    nbr_drop_threshold=suggested['nbr_drop_threshold'],
)

Map.addLayer(proxy_mask.selfMask(), {'palette': ['red']}, 'Pseudo-etiqueta: posible tala selectiva')
Map

**Siguiente paso manual importante**: inspecciona visualmente varios clusters de
esta pseudo-etiqueta contra el RGB. Digitaliza a mano (en este mapa con
`geemap.Map.draw` o en QGIS) un subconjunto de ~30-60 polígonos donde el patrón
de tala selectiva sea inequívoco (trochas, claros de dosel). Ese subconjunto será
tu **test set validado por fotointerpretación** — ver `src/labels.py` sección 3
y el README ("Caveat sobre ground truth").

## 7. Grilla de exportación (parches de 256x256 px)

In [ ]:
tiles = build_export_grid(cfg, roi)
print(f"{len(tiles)} tiles generados de {cfg['tiling']['patch_size_px']}px a {cfg['tiling']['scale_m']}m")
tiles[:3]

## 8. Export de PRUEBA (2-3 tiles) a Google Drive
Antes de exportar TODO el ROI (puede ser docenas de tareas), prueba con pocos
tiles para validar que el formato/bandas son correctos.

In [ ]:
from src.gee_exporter import export_grid_to_drive

full_image = stack.addBands(change)
test_tiles = tiles[:2]
tasks = export_grid_to_drive(cfg, full_image, test_tiles, name_prefix='test_export')
print(f'{len(tasks)} tareas de exportación lanzadas. Revisa el progreso en:')
print('https://code.earthengine.google.com/tasks')